# Assignment 8

In [ ]:
#Instalamos lo necesario
!pip install geopandas matplotlib
!pip install folium

In [ ]:
#Importamos lo necesario
import pandas as pd
from pandas import Series, DataFrame
import numpy as np
import matplotlib.pyplot as plt 
import chardet
import matplotlib.patches as mpatches

In [ ]:
import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString
import folium 
from folium import Marker, GeoJson
from folium.plugins import MarkerCluster, HeatMap

### **1. Import the data located at this [link](https://github.com/alexanderquispe/Diplomado_PUCP/blob/main/_data/data_dengue_peru.csv). It has information on people infected with dengue at the district level for 2015 to 2021.**

In [ ]:
# Importando la data
df_dengue = pd.read_csv(r"../../_data/data_dengue_peru.csv")

In [ ]:
df_dengue

### **2. Generate ubigeo for Departments and Provinces taking the first two and four numbers.**

In [ ]:
# Primero, creamos una nueva variable string 'ubigeo_1' a partir de la variable 'ubigeo' que es int. 
df_dengue['ubigeo_1'] = df_dengue['Ubigeo'].astype(str)

In [ ]:
# Antes de extraer los digitos para crear el ubigeo del departamento y provincia, debemos de asegurarnos que la variable 
# ubigeo contenga 6 digitos.
# Para ello, generamos una función en la que adicione un cero al inicio del ubigeo si dicha variable ubigeo tiene solo 5 digitos
# y que mantenga el ubigeo si es que tiene los 6 digitos.
def seis_digitos(ubigeo):
    if len(ubigeo) == 5:
        return "0" + ubigeo
    else:
        return ubigeo

In [ ]:
# Aplicando la funcion a la variable ubigeo_1 de la base df_dengue
df_dengue['ubigeo_1'] = df_dengue['ubigeo_1'].apply(seis_digitos)

In [ ]:
# viendo la variable ubigeo_1. Se observa que para las observaciones que tenian 5 digitos se les agregó un cero al inicio para así
# hacer que la variable tenga 6 digitos.
df_dengue['ubigeo_1']

In [ ]:
# Creando la variable ubigeo_depa con los dos primeros dos digitos de la variable ubigeo_1
df_dengue['ubigeo_depa'] = df_dengue['ubigeo_1'].str[:2]

# Creando la variable ubigeo_prov con los dos primeros dos digitos de la variable ubigeo_1
df_dengue['ubigeo_prov'] = df_dengue['ubigeo_1'].str[:4]

In [ ]:
# Viendo la variable ubigeo_depa. Como se puede observar, esta variable ubigeo_depa muestra los dos primeros digitos de la
# variable ubigeo_1 asociadas al departamento.
df_dengue['ubigeo_depa']

In [ ]:
# Viendo la variable ubigeo_prov. Como se percibe, esta variable ubigeo_prov muestra los cuatro primeros digitos 
# de la variable ubigeo_1 relacionados a la provincia.
df_dengue['ubigeo_prov']

In [ ]:
df_dengue

### **3. Use geopandas to plot the number of cases in 2021 by the district using a continuous legend.**

In [ ]:
#Primero modificamos la data que contiene los casos de dengue 
df1_dengue=df_dengue.drop("Ubigeo", axis=1)
df1_dengue = df1_dengue.rename({'ubigeo_1':'UBIGEO'}, axis =1)
df1_dengue = df1_dengue[df1_dengue['Año'] == 2021]

In [ ]:
df1_dengue.head(10)

In [ ]:
#Importamos el shape file a nivel de distrito
maps = gpd.read_file(r'../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')
maps.head(2)

In [ ]:
#Seleccionamos las columnas mas relevantes  
maps1 = maps[['UBIGEO', 'geometry']]
maps1.head(5)

In [ ]:
#Hacemos el merge del shape file con la base de casos
df_m = pd.merge(maps1, df1_dengue, on="UBIGEO")
df_m.head(5)

In [ ]:
#Procedemos a reemplazar los datos NA con -1, para ello verificamos si el valor no es una cadena de texto
df_m['Casos'] = df_m['Casos'].apply(lambda x: x.replace(',', '') if isinstance(x, str) else x)
df_m['Casos'] = df_m['Casos'].fillna(-1).astype(int)
df_m.head(3)

In [ ]:
#Elimino las filas que tienen -1 en el la columna "Casos"
df_m = df_m[df_m['Casos'] >= 0]
df_m.head(3)

In [ ]:
#Procedo a hacer un collapse para mantener una sola observación por distrito sumando los casos de las semanas

# Crear un diccionario con las operaciones para cada columna
operations = {col: 'first' for col in df_m.columns if col != 'UBIGEO'}
operations['Casos'] = 'sum'  # La columna 'Casos' se sumará

# Sumar los casos de dengue por distrito y mantener las demás columnas
df_coll = df_m.groupby('UBIGEO').agg(operations).reset_index()
df_coll

#Borro columna semana
df_coll=df_coll.drop("Semana", axis=1)

df_coll

In [ ]:
#Procedemos a volver a mergear la data df_coll con el shapefile maps1, para tener NA en la columna "Casos" para aquellos
#distritos que no aparecen en la data df_coll
df_coll1 = pd.merge(maps1, df_coll, on="UBIGEO", how='left')
df_coll1=df_coll1.drop("geometry_y",axis=1)
df_coll1=df_coll1.rename({'geometry_x':'geometry'}, axis =1 )
df_coll1

In [ ]:
# Ahora bien, antes de realizar el gráfico verificamos la distribución de nuestra serie 
fig, ax = plt.subplots(figsize=(10, 10))
df_coll1["Casos"].hist(bins = 100)

In [ ]:
# Finalmente, realizamos el gráfico 
df_coll1 = df_coll1.set_geometry('geometry')
df_coll1.plot( column='Casos', cmap='Greens', 
          figsize=(20, 20),
          linestyle='-',
          edgecolor='gray',
          legend = True,
             missing_kwds={
                 "color": "lightgrey", # Este color representa un gris suave
                 "edgecolor": "gray",
             })

# Creamos la leyenda
missing_patch = mpatches.Patch(color='lightgrey', label='Missing districts')
plt.legend(handles=[missing_patch])

# Desactivamos los ejes
plt.axis('off')
plt.xticks([])
plt.yticks([])

plt.show()

**4. Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values**

In [ ]:
#Utilizamos la data de casos de dengue modificada, para cambiar el nombre de la columna del ubigeo de provincia 
df2_dengue=df1_dengue
df2_dengue = df2_dengue.rename({'ubigeo_prov':'IDPROV'}, axis =1)
df2_dengue.head(3)

In [ ]:
#Modificamos el shapefile de los distritos para poder accceder a las geometrias a nivel de provincia
prov_shp = maps.dissolve( by = 'IDPROV' )
prov_shp = prov_shp.reset_index()
prov_shp.head(3)

In [ ]:
#Seleccionamos las columnas mas relevantes  
maps2 = prov_shp[['IDPROV', 'geometry']]
maps2.head(5)

In [ ]:
#Hacemos el merge del nuevo shape file con la base de casos
df_m2 = pd.merge(maps2, df2_dengue, on="IDPROV")
df_m2.head(3)

In [ ]:
#Procedemos a reemplazar los datos NA con -1, para ello verificamos si el valor no es una cadena de texto
df_m2['Casos'] = df_m2['Casos'].apply(lambda x: x.replace(',', '') if isinstance(x, str) else x)
df_m2['Casos'] = df_m2['Casos'].fillna(-1).astype(int)
df_m2.head(3)

In [ ]:
#Elimino las filas que tienen -1 en el la columna "Casos"
df_m2 = df_m2[df_m2['Casos'] >= 0]
df_m2.head(3)

In [ ]:
#Procedo a hacer un collapse para mantener una sola observación por distrito sumando los casos de las semanas

# Crear un diccionario con las operaciones para cada columna
operations = {col: 'first' for col in df_m2.columns if col != 'IDPROV'}
operations['Casos'] = 'sum'  # La columna 'Casos' se sumará

# Sumar los casos de dengue por distrito y mantener las demás columnas
df2_coll = df_m2.groupby('IDPROV').agg(operations).reset_index()
df2_coll

#Borro columna semana
df2_coll=df2_coll.drop("Semana", axis=1)

df2_coll.head(3)

In [ ]:
#Procedemos a volver a mergear la data df_coll con el shapefile maps1, para tener NA en la columna "Casos" para aquellos
#distritos que no aparecen en la data df_coll
df2_coll1 = pd.merge(maps2, df2_coll, on="IDPROV", how='left')
df2_coll1=df2_coll1.drop("geometry_y",axis=1)
df2_coll1=df2_coll1.rename({'geometry_x':'geometry'}, axis =1 )
df2_coll1

In [ ]:
# Ahora bien, antes de realizar el gráfico verificamos la distribución de nuestra serie 
fig, ax = plt.subplots(figsize=(10, 10))
df2_coll1["Casos"].hist(bins = 100)

In [ ]:
# Finalmente, realizamos el gráfico 
df2_coll1 = df2_coll1.set_geometry('geometry')
df2_coll1.plot( column='Casos', cmap='Greens', 
          figsize=(20, 20),
          linestyle='-',
          edgecolor='gray',
          legend = True,
             missing_kwds={
                 "color": "lightgrey", # Este color representa un gris suave
                 "edgecolor": "gray",
             })

# Creamos la leyenda
missing_patch = mpatches.Patch(color='lightgrey', label='Missing provincies')
plt.legend(handles=[missing_patch])

# Desactivamos los ejes
plt.axis('off')
plt.xticks([])
plt.yticks([])

plt.show()

**5. Use geopandas to plot the number of cases by the department for all the years using subplots. Every subplot for each year.**

In [ ]:
#Utilizamos la data de casos de dengue para cambiar el nombre de la columna del ubigeo de departamento
df3_dengue=df_dengue
df3_dengue = df_dengue.rename({'ubigeo_depa':'CCDD'}, axis =1)
df3_dengue.head(3)

In [ ]:
#Modificamos el shapefile de los distritos para poder accceder a las geometrias a nivel de departamento
dep_shp = maps.dissolve( by = 'CCDD' )
dep_shp = dep_shp.reset_index()
#Seleccionamos las columnas mas relevantes  
maps3 = dep_shp[['CCDD', 'geometry']]
maps3

In [ ]:
#Hacemos el merge del nuevo shape file con la base de casos
df_m3 = pd.merge(maps3, df3_dengue, on="CCDD")
df_m3

In [ ]:
#Procedemos a reemplazar los datos NA con -1, para ello verificamos si el valor no es una cadena de texto
df_m3['Casos'] = df_m3['Casos'].apply(lambda x: x.replace(',', '') if isinstance(x, str) else x)
df_m3['Casos'] = df_m3['Casos'].fillna(-1).astype(int)

#Elimino las filas que tienen -1 en el la columna "Casos"
df_m3 = df_m3[df_m3['Casos'] >= 0]
df_m3.head(3)

In [ ]:
operations = {col: 'first' for col in df_m3.columns if col != 'CCDD' and col != 'Año'}
operations['Casos'] = 'sum'  # La columna 'Casos' se sumará

# Sumar los casos de dengue por distrito y año, manteniendo las demás columnas
df3_coll = df_m3.groupby(['CCDD', 'Año']).agg(operations).reset_index()

# Borrar la columna 'Semana' si existe
if 'Semana' in df3_coll.columns:
    df3_coll = df3_coll.drop("Semana", axis=1)

df3_coll

In [ ]:
ccdd_unique = df3_coll['CCDD'].unique()
print(ccdd_unique)

In [ ]:
#Procedemos a volver a mergear la data df_coll con el shapefile maps1, para tener NA en la columna "Casos" para aquellos departamentos que no aparecen en la data df_coll
df3_coll1 = pd.merge(maps3, df3_coll, on="CCDD", how='left')
df3_coll1=df3_coll1.drop("geometry_y",axis=1)
df3_coll1=df3_coll1.rename({'geometry_x':'geometry'}, axis =1 )
df3_coll1

In [ ]:
ccdd_unique = df3_coll1['CCDD'].unique()
print(ccdd_unique)

In [ ]:
# Verificamos la distribución de nuestra serie 

# Obtener los años únicos excluyendo missings
unique_years = df3_coll1['Año'].dropna().unique()

# Se establece el número de filas y columnas para los subplots
num_rows = 4
num_cols = 2

# Crear subplots para cada año
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(20, 20), sharey=True)

# Iterar sobre cada año y dibujar un histograma en cada subplot
for i, year in enumerate(unique_years):
    row = i // num_cols  # Calcular el índice de la fila
    col = i % num_cols   # Calcular el índice de la columna
    
    ax = axes[row, col]  # Obtener el subplot correspondiente
    df_year = df3_coll1[df3_coll1['Año'] == year]
    df_year["Casos"].hist(bins=100, ax=ax)
    ax.set_title(f'Año {int(year)}')
    ax.set_xlabel('Casos')
    ax.set_ylabel('Frecuencia')

# Eliminar subplots no utilizados
for i in range(len(unique_years), num_rows*num_cols):
    axes.flatten()[i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Obtener los años únicos en tus datos
unique_years = df3_coll1['Año'].dropna().unique()

# Calcular el número de filas y columnas para los subplots
num_rows = 4
num_cols = 2

# Crear subplots para cada año
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(20, 20))

# Iterar sobre cada año y dibujar el mapa correspondiente
for i, year in enumerate(unique_years):
    row = i // num_cols  # Calcular el índice de la fila
    col = i % num_cols   # Calcular el índice de la columna
    
    ax = axes[row, col]  # Obtener el eje correspondiente
    ax.set_title(f'Año {int(year)}')  # Establecer el título del subplot sin decimales
    df_year = df3_coll1.set_geometry('geometry')
    df_year.plot(column='Casos', cmap='Greens', ax=ax, linestyle='-', edgecolor='gray', legend=True,
                 missing_kwds={"color": "lightgrey", "edgecolor": "gray"})
    ax.axis('off')  # Desactivar los ejes
    ax.set_aspect('equal')  # Establecer el aspecto del eje como igual

# Crear la leyenda
missing_patch = mpatches.Patch(color='lightgrey', label='Missing provinces')
plt.legend(handles=[missing_patch])

# Eliminar subplots no utilizados
for i in range(len(unique_years), num_rows*num_cols):
    axes.flatten()[i].axis('off')

plt.tight_layout()
plt.show()

6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.


In [ ]:
pip install mapclassify


In [ ]:
# Utilizamos la data de casos de dengue ya filtrada para 2021
df3_dengue = df_dengue[df_dengue['Año'] == 2021].copy()

# Renombrar 'ubigeo_depa' a 'CCDD' para coincidir con el shapefile a nivel de departamento
df3_dengue = df3_dengue.rename({'ubigeo_depa':'CCDD'}, axis=1)

# Asegurar que las columnas 'Semana' y 'Casos' sean numéricas
df3_dengue['Semana'] = pd.to_numeric(df3_dengue['Semana'], errors='coerce')
df3_dengue['Casos'] = pd.to_numeric(df3_dengue['Casos'], errors='coerce')

# Llenar los valores NaN resultantes con 0 o un valor apropiado
df3_dengue.fillna({'Semana': 0, 'Casos': 0}, inplace=True)

# Determinar el trimestre de cada caso
df3_dengue['Trimestre'] = pd.cut(df3_dengue['Semana'], bins=[1, 13, 26, 39, 53], labels=['Q1', 'Q2', 'Q3', 'Q4'], right=False)

# Agrupar los casos por departamento y trimestre
df3_dengue_grouped = df3_dengue.groupby(['CCDD', 'Trimestre']).agg({'Casos': 'sum'}).reset_index()

# Utilizamos el shapefile de departamentos ya definido anteriormente (maps3)
df3_dengue_merged = pd.merge(maps3, df3_dengue_grouped, on='CCDD', how='left')

# Reemplazar los valores NA con -1 para indicar los departamentos sin datos
df3_dengue_merged['Casos'].fillna(-1, inplace=True)

# Configuración de la visualización para cada trimestre
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 20), tight_layout=True)
trimestres = ['Q1', 'Q2', 'Q3', 'Q4']

for i, ax in enumerate(axes.flatten()):
    trimestre_data = df3_dengue_merged[df3_dengue_merged['Trimestre'] == trimestres[i]]
    trimestre_data.plot(column='Casos', ax=ax, legend=True, cmap='OrRd', scheme='Quantiles', k=5,
                        missing_kwds={'color': 'lightgrey', 'edgecolor': 'black', 'label': 'No data'})
    ax.set_title(f'Casos de Dengue por Departamento - {trimestres[i]} 2021')
    ax.axis('off')

plt.show()
